[![Open In Colab](https://colab.research.google.com/assets/colab-badge.png)](https://colab.research.google.com/github/lgonperez3-glitch/MLSM-labs/blob/main/Lab3/06_01_autoencoders_mtat_v5.ipynb)

# Lecture 6: Self-supervised learning and autoencoders

## MagnaTagATune — 1-second segments

This notebook compares two trained autoencoders: a three-block baseline optimized with waveform MSE and a five-block improved autoencoder optimized with a combined MSE + multi-resolution STFT objective. For each run, only the checkpoint from the epoch with the lowest validation loss is saved. Both runs use the same MTAT split, 30% training subset, full validation set, 20 epochs, 16 kHz one-second segments, and a 31.25:1 bottleneck compression rate.

In [16]:
# Install deps

!pip install torchcodec
!pip install torch
!pip install mirdata
!pip install tqdm
!pip install scikit-learn
!pip install torchaudio
!pip install matplotlib
!pip install umap-learn
!pip install auraloss torch-audiomentations

In [17]:
import os
import json
import shutil
import subprocess
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import umap
import auraloss
from torch_audiomentations import Compose, Gain, AddColoredNoise, BandStopFilter, PitchShift, LowPassFilter

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm, trange
from IPython.display import Audio

# Set to cuda or cpu, depending on the runtime
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


## Dataset preparation — MagnaTagATune

The dataset is split by original recording into training and validation subsets (80/20) before sampling. A reproducible 30% subset of the training recordings is used; the validation set remains unchanged. Each sample is one randomly selected 1-second segment from a recording.


In [18]:
# Mount Google Drive and define the requested results folder
from google.colab import drive
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount('/content/drive')
else:
    print("Google Drive is already mounted.")

RESULTS_DIR = "/content/drive/MyDrive/MLSM/Lab 3/results_v5"
os.makedirs(RESULTS_DIR, exist_ok=True)

# MagnaTagATune files hosted by City, University of London
MTAT_ROOT = "/content/mtat"
AUDIO_DIR = os.path.join(MTAT_ROOT, "mp3")
ANNOTATIONS_PATH = os.path.join(MTAT_ROOT, "annotations_final.csv")

os.makedirs(MTAT_ROOT, exist_ok=True)

# Resume interrupted downloads and verify each file against the server's file size.
import urllib.request

def download_with_resume(url, destination):
    request = urllib.request.Request(url, method="HEAD")
    with urllib.request.urlopen(request, timeout=30) as response:
        expected_size = int(response.headers.get("Content-Length", 0))

    if os.path.exists(destination) and expected_size and os.path.getsize(destination) > expected_size:
        os.remove(destination)

    subprocess.run(["wget", "-c", "--tries=5", "--timeout=30", url, "-P", MTAT_ROOT], check=True)

    downloaded_path = os.path.join(MTAT_ROOT, os.path.basename(url))
    if not os.path.exists(downloaded_path):
        raise FileNotFoundError(f"Download did not create: {downloaded_path}")

    actual_size = os.path.getsize(downloaded_path)
    if expected_size and actual_size != expected_size:
        raise RuntimeError(
            f"Incomplete download for {os.path.basename(url)}: "
            f"{actual_size:,} bytes downloaded, expected {expected_size:,}. "
            "Run this cell again; wget will resume the download."
        )
    if actual_size == 0:
        raise RuntimeError(f"Downloaded file is empty: {downloaded_path}")

    return downloaded_path

# Check annotations on every run; wget -c skips a complete file and resumes a partial one.
download_with_resume(
    "https://mirg.city.ac.uk/datasets/magnatagatune/annotations_final.csv",
    ANNOTATIONS_PATH
)

# The archive contains 16 shards (0-9 and a-f), extracted inside AUDIO_DIR.
expected_shards = set("0123456789abcdef")
os.makedirs(AUDIO_DIR, exist_ok=True)

# Migrate shards if a previous run extracted them one directory too high.
for shard in expected_shards:
    old_location = os.path.join(MTAT_ROOT, shard)
    new_location = os.path.join(AUDIO_DIR, shard)
    if os.path.isdir(old_location) and not os.path.exists(new_location):
        shutil.move(old_location, new_location)

existing_shards = set(os.listdir(AUDIO_DIR))
audio_is_complete = expected_shards.issubset(existing_shards)

if not audio_is_complete:
    archive_parts = ["mp3.zip.001", "mp3.zip.002", "mp3.zip.003"]
    for part in archive_parts:
        download_with_resume(
            f"https://mirg.city.ac.uk/datasets/magnatagatune/{part}",
            os.path.join(MTAT_ROOT, part)
        )

    # Rebuild the joined archive from the verified parts.
    archive_path = os.path.join(MTAT_ROOT, "mp3.zip")
    with open(archive_path, "wb") as output_file:
        for part in archive_parts:
            part_path = os.path.join(MTAT_ROOT, part)
            with open(part_path, "rb") as input_file:
                shutil.copyfileobj(input_file, output_file)

    # The official archive contains folders 0-f at its root: extract them into AUDIO_DIR.
    subprocess.run(["unzip", "-q", "-o", archive_path, "-d", AUDIO_DIR], check=True)

    extracted_shards = set(os.listdir(AUDIO_DIR))
    missing_shards = sorted(expected_shards - extracted_shards)
    if missing_shards:
        raise RuntimeError(
            f"MTAT extraction is incomplete. Missing audio folders: {missing_shards}. "
            f"Top-level entries found: {sorted(extracted_shards)[:30]}"
        )

print("Audio directory:", AUDIO_DIR)
print("Annotations file:", ANNOTATIONS_PATH)
print("Audio directory exists:", os.path.isdir(AUDIO_DIR))


Google Drive is already mounted.
Audio directory: /content/mtat/mp3
Annotations file: /content/mtat/annotations_final.csv
Audio directory exists: True


In [19]:
# Read MTAT annotations
annotations = pd.read_csv(ANNOTATIONS_PATH, sep="\t")

# Exclude three empty MP3 files
invalid_files = {
    "6/norine_braun-now_and_zen-08-gently-117-146.mp3",
    "8/jacob_heringman-josquin_des_prez_lute_settings-19-gintzler__pater_noster-204-233.mp3",
    "9/american_baroque-dances_and_suites_of_rameau_and_couperin-25-le_petit_rien_xiveme_ordre_couperin-88-117.mp3",
}

annotations = annotations[
    ~annotations["mp3_path"].isin(invalid_files)
].copy()

# Identify metadata columns and tag columns
metadata_columns = [column for column in ["clip_id", "mp3_path"] if column in annotations.columns]
tag_columns = [column for column in annotations.columns if column not in metadata_columns]

# Keep rows with an available audio path and at least one positive tag
annotations = annotations.dropna(subset=["mp3_path"]).copy()
annotations[tag_columns] = annotations[tag_columns].apply(pd.to_numeric, errors="coerce").fillna(0)
annotations = annotations.loc[annotations[tag_columns].sum(axis=1) > 0].reset_index(drop=True)

# Use the most strongly annotated tag as the display label for UMAP
annotations["natural_label"] = annotations[tag_columns].idxmax(axis=1)

# Split by original recording, not by segment, to avoid data leakage
train_rows, val_rows = train_test_split(
    np.arange(len(annotations)),
    test_size=0.2,
    random_state=42,
    shuffle=True
)

train_annotations = annotations.iloc[train_rows].reset_index(drop=True)
val_annotations = annotations.iloc[val_rows].reset_index(drop=True)

# Use the same 30% training-data subsampling criterion as the Lab 4 reference.
# Keep the validation set unchanged and sample reproducibly.
full_train_size = len(train_annotations)
train_annotations = train_annotations.sample(
    n=int(full_train_size * 0.3),
    random_state=42
).reset_index(drop=True)

print(f"Tracks: {len(annotations)}")
print(f"Training tracks before subsampling: {full_train_size}")
print(f"Training tracks used (30%): {len(train_annotations)}")
print(f"Validation tracks (100%): {len(val_annotations)}")
print(f"Number of tags: {len(tag_columns)}")


Tracks: 21639
Training tracks before subsampling: 17311
Training tracks used (30%): 5193
Validation tracks (100%): 4328
Number of tags: 188


In [20]:
class MTATSegmentDataset(Dataset):
    """MTAT dataset returning one random 1-second segment per recording."""

    def __init__(
        self,
        annotations,
        audio_root,
        sample_rate=16000,
        segment_duration=1
    ):
        self.annotations = annotations.reset_index(drop=True)
        self.audio_root = audio_root
        self.sample_rate = sample_rate
        self.segment_duration = segment_duration
        self.segment_samples = sample_rate * segment_duration

    def __len__(self):
        # One sample per recording in each epoch.
        return len(self.annotations)

    def __getitem__(self, idx):
        row = self.annotations.iloc[idx]
        audio_path = os.path.join(self.audio_root, str(row["mp3_path"]))

        # Load and decode the MP3 every time a sample is requested.
        audio, sr = torchaudio.load(audio_path)

        # Convert to mono.
        if audio.shape[0] > 1:
            audio = audio.mean(dim=0, keepdim=True)

        # Resample to 16 kHz when necessary.
        if sr != self.sample_rate:
            resampler = torchaudio.transforms.Resample(sr, self.sample_rate)
            audio = resampler(audio)

        audio = audio.squeeze(0)

        # Normalize the complete recording.
        max_val = torch.abs(audio).max()
        if max_val > 0:
            audio = audio / max_val

        # Select one random 1-second segment from the first 27 seconds.
        max_start_second = max(
            0,
            int(np.ceil(len(audio) / self.sample_rate)) - self.segment_duration
        )
        max_start_second = min(max_start_second, 26)
        start_second = torch.randint(0, max_start_second + 1, (1,)).item()

        start = start_second * self.sample_rate
        end = start + self.segment_samples
        segment = audio[start:end]

        if len(segment) < self.segment_samples:
            segment = F.pad(segment, (0, self.segment_samples - len(segment)))

        return {
            "audio": segment,
            "natural_labels": row["natural_label"],
            "track_id": str(row["clip_id"]) if "clip_id" in row else str(idx)
        }


train_dataset = MTATSegmentDataset(train_annotations, AUDIO_DIR)
val_dataset = MTATSegmentDataset(val_annotations, AUDIO_DIR)

train_loader = DataLoader(
    train_dataset, batch_size=128, shuffle=True, num_workers=2,
    pin_memory=(device == "cuda")
)
val_loader = DataLoader(
    val_dataset, batch_size=128, shuffle=False, num_workers=2,
    pin_memory=(device == "cuda")
)

print(f"Training recordings: {len(train_dataset)}")
print(f"Validation recordings: {len(val_dataset)}")
print(f"Segment length: {train_dataset.segment_samples} samples ({train_dataset.segment_duration} second)")
print("Audio storage: loaded on demand; one random segment per recording; no preload or cache")


Training recordings: 5193
Validation recordings: 4328
Segment length: 16000 samples (1 second)
Audio storage: loaded on demand; one random segment per recording; no preload or cache


# Model and training strategy

Two checkpoints are saved, one for each trained model:

1. **Baseline:** three encoder/decoder blocks, no BatchNorm, optimized with waveform MSE. Its validation loss is MSE because the MR-STFT coefficient is zero.
2. **Improved:** five encoder/decoder blocks, BatchNorm, optimized with $0.1\,L_{MSE}+1.0\,L_{MR-STFT}$.

For both runs, the saved checkpoint is the epoch with the lowest validation loss. The improved model's validation loss is the weighted sum of validation MSE and validation MR-STFT. The baseline's validation loss is validation MSE. All variants preserve the 512-dimensional bottleneck (31.25:1 compression), training data, validation split, augmentation policy, batch size, optimizer, and learning rate.

In [21]:
# Shared training configuration
target_length = 16000
bottleneck_size = 512
downsample = 2
num_epochs = 20
COMPRESSION_RATE = target_length / bottleneck_size
assert abs(COMPRESSION_RATE - 31.25) < 1e-8
print(f"Compression rate: {COMPRESSION_RATE:.2f}:1")

class AugmentationChain(nn.Module):
    """Audio augmentation pipeline adapted from the Lab 4 reference notebook."""
    def __init__(self):
        super().__init__()
        self.chain = Compose(
            transforms=[
                Gain(p=0.75, output_type="tensor"),
                BandStopFilter(sample_rate=16000, p=0.75, output_type="tensor"),
                PitchShift(p=0.75, sample_rate=16000, output_type="tensor"),
                AddColoredNoise(p=0.75, output_type="tensor"),
                LowPassFilter(
                    p=0.75, sample_rate=16000,
                    min_cutoff_freq=400, output_type="tensor"
                ),
            ],
            output_type="tensor"
        )

    def forward(self, audio):
        return self.chain(audio.unsqueeze(1)).squeeze(1)


def train_autoencoder(
    model,
    device,
    train_loader,
    val_loader,
    num_epochs=20,
    learning_rate=1e-3,
    use_augmentation=True,
    alpha=1.0,
    beta=0.0,
    checkpoint_paths=None,
    run_name="model"
):
    """
    Train an autoencoder and save the checkpoint with the lowest validation
    objective, alpha*MSE+beta*MR-STFT.
    """
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    mse_criterion = nn.MSELoss()

    perceptual_criterion = auraloss.freq.MultiResolutionSTFTLoss(
        fft_sizes=[512, 1024, 2048],
        hop_sizes=[128, 256, 512],
        win_lengths=[512, 1024, 2048],
        scale="mel",
        n_bins=128,
        sample_rate=16000,
        perceptual_weighting=True,
    ).to(device)

    augmentation = AugmentationChain().to(device) if use_augmentation else None
    if augmentation is not None:
        augmentation.eval()
        print("Data augmentation enabled")

    history = {
        "train_loss": [], "val_loss": [],
        "train_mse": [], "train_perceptual": [],
        "val_mse": [], "val_perceptual": []
    }
    best_values = {"validation_loss": float("inf")}
    best_epochs = {"validation_loss": 0}

    for epoch in range(num_epochs):
        model.train()
        train_losses, train_mse_losses, train_perceptual_losses = [], [], []
        train_pbar = tqdm(train_loader, desc=f"{run_name} — Epoch {epoch+1}/{num_epochs} [Train]")

        for batch in train_pbar:
            audio = batch["audio"].to(device)
            if augmentation is not None:
                with torch.no_grad():
                    audio = augmentation(audio)

            optimizer.zero_grad()
            recon_audio, _ = model(audio.unsqueeze(1))
            recon_audio = recon_audio.squeeze(1)

            mse_value = mse_criterion(recon_audio, audio)
            perceptual_value = perceptual_criterion(
                recon_audio.unsqueeze(1), audio.unsqueeze(1)
            )
            loss = alpha * mse_value + beta * perceptual_value
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())
            train_mse_losses.append(mse_value.item())
            train_perceptual_losses.append(perceptual_value.item())
            train_pbar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "mse": f"{mse_value.item():.4f}",
                "mr_stft": f"{perceptual_value.item():.4f}"
            })

        model.eval()
        val_losses, val_mse_losses, val_perceptual_losses = [], [], []
        with torch.no_grad():
            val_pbar = tqdm(val_loader, desc=f"{run_name} — Epoch {epoch+1}/{num_epochs} [Val]")
            for batch in val_pbar:
                audio = batch["audio"].to(device)
                recon_audio, _ = model(audio.unsqueeze(1))
                recon_audio = recon_audio.squeeze(1)

                mse_value = mse_criterion(recon_audio, audio)
                perceptual_value = perceptual_criterion(
                    recon_audio.unsqueeze(1), audio.unsqueeze(1)
                )
                val_loss = alpha * mse_value + beta * perceptual_value

                val_losses.append(val_loss.item())
                val_mse_losses.append(mse_value.item())
                val_perceptual_losses.append(perceptual_value.item())
                val_pbar.set_postfix({
                    "loss": f"{val_loss.item():.4f}",
                    "mse": f"{mse_value.item():.4f}",
                    "mr_stft": f"{perceptual_value.item():.4f}"
                })

        history["train_loss"].append(float(np.mean(train_losses)))
        history["train_mse"].append(float(np.mean(train_mse_losses)))
        history["train_perceptual"].append(float(np.mean(train_perceptual_losses)))
        history["val_loss"].append(float(np.mean(val_losses)))
        history["val_mse"].append(float(np.mean(val_mse_losses)))
        history["val_perceptual"].append(float(np.mean(val_perceptual_losses)))

        val_mse = history["val_mse"][-1]
        val_mr_stft = history["val_perceptual"][-1]
        print(
            f"Epoch {epoch+1}: train_loss={history['train_loss'][-1]:.6f}, "
            f"val_loss={history['val_loss'][-1]:.6f}, "
            f"val_MSE={val_mse:.6f}, val_MR_STFT={val_mr_stft:.6f}"
        )

        val_loss = history["val_loss"][-1]
        if val_loss < best_values["validation_loss"]:
            best_values["validation_loss"] = val_loss
            best_epochs["validation_loss"] = epoch + 1
            checkpoint_path = (checkpoint_paths or {}).get("validation_loss")
            if checkpoint_path:
                torch.save({
                    "epoch": epoch + 1,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "selection_metric": "validation_loss",
                    "selection_value": val_loss,
                    "val_loss": val_loss,
                    "val_mse": val_mse,
                    "val_mr_stft": val_mr_stft,
                    "history": history,
                    "config": {
                        "alpha": alpha,
                        "beta": beta,
                        "use_augmentation": use_augmentation,
                        "learning_rate": learning_rate,
                        "num_epochs": num_epochs,
                        "selection_metric": "validation_loss"
                    }
                }, checkpoint_path)
                print(
                    f"Saved best validation-loss checkpoint at epoch {epoch+1} "
                    f"({val_loss:.6f})"
                )

    # Restore the state with the lowest validation loss.
    best_checkpoint = (checkpoint_paths or {}).get("validation_loss")
    if best_checkpoint and os.path.exists(best_checkpoint):
        checkpoint = torch.load(best_checkpoint, map_location=device)
        model.load_state_dict(checkpoint["model_state_dict"])
        print(
            f"Restored best validation-loss checkpoint from epoch {checkpoint['epoch']} "
            f"({checkpoint['val_loss']:.6f})"
        )

    history["best_epochs"] = best_epochs
    history["best_values"] = best_values
    with open(os.path.join(RESULTS_DIR, f"{run_name}_training_history.json"), "w") as file:
        json.dump(history, file, indent=2)

    return history

Compression rate: 31.25:1


# Baseline and improved architectures

Both variants use the same mirrored convolution / max-pooling-index / max-unpool structure and the same 512-dimensional bottleneck. The baseline has three encoder/decoder blocks and no BatchNorm. The improved model has two additional blocks (five total) and BatchNorm in its intermediate decoder blocks.

In [22]:
class AudioAutoencoderImproved(nn.Module):
    """
    1-D convolutional autoencoder for 1-second MTAT segments.
    extra_layers=0 gives the three-block baseline; extra_layers=2 gives
    the five-block improved model. BatchNorm can be disabled for the baseline.
    """
    def __init__(
        self,
        input_length=16000,
        bottleneck_size=512,
        kernel_size=5,
        channels=8,
        channel_mult=2,
        downsample=2,
        extra_layers=2,
        use_batchnorm=True,
    ):
        super().__init__()
        assert kernel_size % 2 == 1, "kernel_size should be odd for same padding"

        self.num_layers = 3 + extra_layers
        self.downsample = downsample
        padding = kernel_size // 2
        self.encoder_convs = nn.ModuleList()
        self.encoder_norms = nn.ModuleList()
        self.encoder_acts = nn.ModuleList()
        self.pools = nn.ModuleList()
        self.channel_sizes = [
            channels * (channel_mult ** layer_idx)
            for layer_idx in range(self.num_layers)
        ]

        in_channels = 1
        for out_channels in self.channel_sizes:
            self.encoder_convs.append(nn.Conv1d(
                in_channels, out_channels, kernel_size=kernel_size, padding=padding
            ))
            self.encoder_norms.append(
                nn.BatchNorm1d(out_channels) if use_batchnorm else nn.Identity()
            )
            self.encoder_acts.append(nn.PReLU())
            self.pools.append(nn.MaxPool1d(
                kernel_size=2, stride=downsample, return_indices=True
            ))
            in_channels = out_channels

        encoded_length = input_length // (downsample ** self.num_layers)
        self.encoded_features = encoded_length * self.channel_sizes[-1]
        self.enc_fc = nn.Linear(self.encoded_features, bottleneck_size)
        print(
            f"Compression rate: {input_length / bottleneck_size:.2f}:1 | "
            f"Encoder/decoder blocks: {self.num_layers} | BatchNorm: {use_batchnorm}"
        )

        self.dec_fc = nn.Linear(bottleneck_size, self.encoded_features)
        self.unpools = nn.ModuleList()
        self.decoder_convs = nn.ModuleList()
        self.decoder_norms = nn.ModuleList()
        self.decoder_acts = nn.ModuleList()

        for layer_idx in reversed(range(self.num_layers)):
            in_channels = self.channel_sizes[layer_idx]
            out_channels = self.channel_sizes[layer_idx - 1] if layer_idx > 0 else 1
            self.unpools.append(nn.MaxUnpool1d(kernel_size=2, stride=downsample))
            self.decoder_convs.append(nn.Conv1d(
                in_channels, out_channels, kernel_size=kernel_size, padding=padding
            ))

            if layer_idx > 0:
                self.decoder_norms.append(
                    nn.BatchNorm1d(out_channels)
                    if use_batchnorm else nn.Identity()
                )
                self.decoder_acts.append(nn.PReLU())
            else:
                self.decoder_norms.append(nn.Identity())
                self.decoder_acts.append(nn.Tanh())

    def encoder(self, x):
        sizes, indices = [], []
        for conv, norm, activation, pool in zip(
            self.encoder_convs, self.encoder_norms, self.encoder_acts, self.pools
        ):
            x = activation(norm(conv(x)))
            sizes.append(x.size())
            x, pooling_indices = pool(x)
            indices.append(pooling_indices)
        z = self.enc_fc(x.flatten(start_dim=1))
        return z, indices, sizes

    def decoder(self, z, indices, sizes):
        x = self.dec_fc(z)
        x = x.view(sizes[-1][0], sizes[-1][1], -1)
        for pooling_indices, output_size, unpool, conv, norm, activation in zip(
            reversed(indices), reversed(sizes), self.unpools,
            self.decoder_convs, self.decoder_norms, self.decoder_acts
        ):
            x = unpool(x, pooling_indices, output_size=output_size)
            x = activation(norm(conv(x)))
        return x

    def forward(self, x):
        z, indices, sizes = self.encoder(x)
        return self.decoder(z, indices, sizes), z

In [ ]:
# Train the baseline and the improved combined-loss model.
# Each run saves only the epoch with the lowest validation loss.
baseline_name = "baseline_3_blocks_best_val_loss"
improved_name = "improved_5_blocks_combined_best_val_loss"

baseline_model = AudioAutoencoderImproved(
    input_length=target_length,
    bottleneck_size=bottleneck_size,
    downsample=downsample,
    extra_layers=0,
    use_batchnorm=False
)
baseline_checkpoint = os.path.join(RESULTS_DIR, f"{baseline_name}.pth")
baseline_history = train_autoencoder(
    baseline_model, device, train_loader, val_loader,
    num_epochs=num_epochs,
    learning_rate=1e-3,
    use_augmentation=True,
    alpha=1.0,
    beta=0.0,
    checkpoint_paths={"validation_loss": baseline_checkpoint},
    run_name=baseline_name
)

improved_model = AudioAutoencoderImproved(
    input_length=target_length,
    bottleneck_size=bottleneck_size,
    downsample=downsample,
    extra_layers=2,
    use_batchnorm=True
)
improved_checkpoint = os.path.join(RESULTS_DIR, f"{improved_name}.pth")
improved_history = train_autoencoder(
    improved_model, device, train_loader, val_loader,
    num_epochs=num_epochs,
    learning_rate=1e-3,
    use_augmentation=True,
    alpha=0.1,
    beta=1.0,
    checkpoint_paths={"validation_loss": improved_checkpoint},
    run_name=improved_name
)

model_results = {
    baseline_name: {
        "model": baseline_model,
        "history": baseline_history,
        "config": {
            "architecture": "3 blocks; BatchNorm disabled",
            "loss": "MSE (validation loss = MSE)",
            "alpha": 1.0,
            "beta": 0.0,
            "selection_metric": "validation_loss",
            "checkpoint": baseline_checkpoint,
            "selected_epoch": baseline_history["best_epochs"]["validation_loss"]
        }
    },
    improved_name: {
        "model": improved_model,
        "history": improved_history,
        "config": {
            "architecture": "5 blocks; BatchNorm enabled",
            "loss": "0.1 * MSE + 1.0 * MR-STFT",
            "alpha": 0.1,
            "beta": 1.0,
            "selection_metric": "validation_loss",
            "checkpoint": improved_checkpoint,
            "selected_epoch": improved_history["best_epochs"]["validation_loss"]
        }
    }
}

print("\nBoth best-validation-loss checkpoints have been saved:")
for name, result in model_results.items():
    print(
        f"{name}: epoch {result['config']['selected_epoch']} | "
        f"validation loss={result['history']['best_values']['validation_loss']:.6f} | "
        f"{result['config']['checkpoint']}"
    )

Compression rate: 31.25:1 | Encoder/decoder blocks: 3 | BatchNorm: False
Data augmentation enabled


baseline_3_blocks_best_val_loss — Epoch 1/20 [Train]: 100%|██████████| 41/41 [01:58<00:00,  2.90s/it, loss=0.0259, mse=0.0259, mr_stft=2.3729]
baseline_3_blocks_best_val_loss — Epoch 1/20 [Val]: 100%|██████████| 34/34 [01:00<00:00,  1.78s/it, loss=0.0223, mse=0.0223, mr_stft=2.4164]


Epoch 1: train_loss=0.031712, val_loss=0.024209, val_MSE=0.024209, val_MR_STFT=2.190759
Saved best validation-loss checkpoint at epoch 1 (0.024209)


baseline_3_blocks_best_val_loss — Epoch 2/20 [Train]: 100%|██████████| 41/41 [01:13<00:00,  1.79s/it, loss=0.0173, mse=0.0173, mr_stft=1.7536]
baseline_3_blocks_best_val_loss — Epoch 2/20 [Val]: 100%|██████████| 34/34 [00:59<00:00,  1.75s/it, loss=0.0163, mse=0.0163, mr_stft=2.0148]


Epoch 2: train_loss=0.020688, val_loss=0.017288, val_MSE=0.017288, val_MR_STFT=1.942313
Saved best validation-loss checkpoint at epoch 2 (0.017288)


baseline_3_blocks_best_val_loss — Epoch 3/20 [Train]: 100%|██████████| 41/41 [01:34<00:00,  2.30s/it, loss=0.0104, mse=0.0104, mr_stft=1.9999]
baseline_3_blocks_best_val_loss — Epoch 3/20 [Val]: 100%|██████████| 34/34 [00:58<00:00,  1.71s/it, loss=0.0113, mse=0.0113, mr_stft=1.8869]


... (54 outputs left out)

Epoch 10: train_loss=1.073013, val_loss=1.041633, val_MSE=0.024383, val_MR_STFT=1.039195
Saved best validation-loss checkpoint at epoch 10 (1.041633)


improved_5_blocks_combined_best_val_loss — Epoch 11/20 [Train]: 100%|██████████| 41/41 [01:30<00:00,  2.21s/it, loss=1.0778, mse=0.0238, mr_stft=1.0754]
improved_5_blocks_combined_best_val_loss — Epoch 11/20 [Val]: 100%|██████████| 34/34 [00:55<00:00,  1.62s/it, loss=1.0332, mse=0.0210, mr_stft=1.0312]


Epoch 11: train_loss=1.040059, val_loss=1.028769, val_MSE=0.022702, val_MR_STFT=1.026499
Saved best validation-loss checkpoint at epoch 11 (1.028769)


improved_5_blocks_combined_best_val_loss — Epoch 12/20 [Train]:  37%|███▋      | 15/41 [00:33<01:07,  2.61s/it, loss=1.0636, mse=0.0199, mr_stft=1.0616]

# Training loss evolution: MSE and MR-STFT

The plots show train and validation MSE and MR-STFT for both training runs. The baseline is optimized with MSE only; the improved model is optimized with the weighted combined objective. The two improved checkpoints differ only in their validation-based selection criterion.

In [ ]:
# Plot the metric histories for the baseline and improved training runs.
histories_to_plot = {
    baseline_name: baseline_history,
    "improved_5_blocks_combined": improved_history
}

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for run_name, history in histories_to_plot.items():
    epochs = range(1, len(history["train_mse"]) + 1)
    axes[0, 0].plot(epochs, history["train_mse"], marker="o", markersize=3, label=run_name)
    axes[0, 1].plot(epochs, history["val_mse"], marker="o", markersize=3, label=run_name)
    axes[1, 0].plot(epochs, history["train_perceptual"], marker="o", markersize=3, label=run_name)
    axes[1, 1].plot(epochs, history["val_perceptual"], marker="o", markersize=3, label=run_name)

titles = [
    "Training MSE", "Validation MSE",
    "Training MR-STFT", "Validation MR-STFT"
]
for ax, title in zip(axes.flat, titles):
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_xticks(range(1, num_epochs + 1))
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "training_metrics_all_runs.png"), dpi=200)
plt.show()

In [ ]:

# Improved Model — Evolution of Validation Loss, MSE and MR-STFT

# Use the metrics already stored during training in improved_history.
improved_epochs = range(1, len(improved_history["val_loss"]) + 1)

fig, ax1 = plt.subplots(figsize=(12, 6))

# Left axis: combined validation loss and MR-STFT
line1 = ax1.plot(
    improved_epochs,
    improved_history["val_loss"],
    marker="o",
    markersize=5,
    linewidth=2,
    label="Validation Loss (0.1 × MSE + MR-STFT)"
)

line2 = ax1.plot(
    improved_epochs,
    improved_history["val_perceptual"],
    marker="s",
    markersize=5,
    linewidth=2,
    label="Validation MR-STFT"
)

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Validation Loss / MR-STFT")
ax1.set_xticks(list(improved_epochs))
ax1.grid(True, alpha=0.3)

# Right axis: validation MSE
ax2 = ax1.twinx()

line3 = ax2.plot(
    improved_epochs,
    improved_history["val_mse"],
    marker="^",
    markersize=5,
    linewidth=2,
    linestyle="--",
    label="Validation MSE"
)

ax2.set_ylabel("Validation MSE")

# Combine the legends from both axes
lines = line1 + line2 + line3
labels = [line.get_label() for line in lines]

ax1.legend(
    lines,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.15),
    ncol=1,
    frameon=False
)

ax1.set_title(
    "Improved Autoencoder — Validation Metrics Across Epochs",
    fontsize=13,
    pad=15
)

fig.tight_layout()

# Save the figure
output_path = os.path.join(
    RESULTS_DIR,
    "improved_validation_metrics_evolution.png"
)

fig.savefig(
    output_path,
    dpi=250,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print(f"Improved validation metrics plot saved: {output_path}")


# UMAP projection of the latent representations

The latent representations of both selected checkpoints are projected using the same validation split.

In [ ]:
all_results = model_results

def plot_umap(model, model_name, max_points=5000):
    model.eval()
    embeddings = []
    natural_labels = []
    collected = 0

    # Extract latent representations from the validation set
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Extracting embeddings: {model_name}"):
            x = batch["audio"].unsqueeze(1).to(device)
            _, z = model(x)

            embeddings.append(z.cpu().numpy())
            natural_labels.extend(batch["natural_labels"])
            collected += len(z)

            if collected >= max_points:
                break

    embeddings = np.concatenate(embeddings, axis=0)[:max_points]
    natural_labels = np.asarray(natural_labels)[:max_points]

    # Compute an independent UMAP projection for this model
    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=15,
        min_dist=0.1,
        random_state=42
    )
    umap_embeddings = reducer.fit_transform(embeddings)

    # Identify all labels
    unique_labels = np.unique(natural_labels)
    n_labels = len(unique_labels)

    # Use a distinct colour for each label
    cmap = plt.get_cmap("turbo", n_labels)

    # Make the figure taller when there are many labels,
    # so the complete legend remains readable
    figure_height = max(8, 0.24 * n_labels + 2)

    fig, ax = plt.subplots(figsize=(14, figure_height))

    legend_handles = []

    # Plot each label separately to create a complete legend
    for label_idx, label in enumerate(unique_labels):
        mask = natural_labels == label
        color = cmap(label_idx)

        scatter = ax.scatter(
            umap_embeddings[mask, 0],
            umap_embeddings[mask, 1],
            color=[color],
            label=str(label),
            s=7,
            alpha=0.75,
            edgecolors="none"
        )

        legend_handles.append(scatter)

    ax.set_title(f"UMAP of Audio Embeddings — {model_name}", fontsize=13)
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    ax.grid(False)

    # Place the complete legend outside the embedding area
    ax.legend(
        handles=legend_handles,
        labels=[str(label) for label in unique_labels],
        title="Most common MTAT tag",
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        fontsize=8,
        title_fontsize=9,
        markerscale=1.8,
        frameon=False,
        borderaxespad=0,
        labelspacing=0.5
    )

    # Reserve space on the right for the legend
    fig.subplots_adjust(
        left=0.08,
        right=0.72,
        top=0.93,
        bottom=0.08
    )

    # Save the figure with the external legend included
    output_path = os.path.join(
        RESULTS_DIR,
        f"umap_{model_name}.png"
    )

    fig.savefig(
        output_path,
        dpi=250,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    print(f"UMAP saved: {output_path}")


# Generate two independent UMAP visualizations
for model_name, result in all_results.items():
    plot_umap(result["model"], model_name)

# Final metrics

In [ ]:
# Save checkpoint-level validation metrics and experiment settings.
summary = []
for model_name, result in all_results.items():
    history = result["history"]
    config = result["config"]
    selection_metric = config["selection_metric"]
    selected_epoch = config["selected_epoch"]
    epoch_index = selected_epoch - 1
    summary.append({
        "model": model_name,
        "architecture": config["architecture"],
        "training_objective": config["loss"],
        "selection_metric": selection_metric,
        "selected_epoch": selected_epoch,
        "selected_validation_mse": history["val_mse"][epoch_index],
        "selected_validation_mr_stft": history["val_perceptual"][epoch_index],
        "selected_validation_objective": history["val_loss"][epoch_index],
        "final_train_mse": history["train_mse"][-1],
        "final_validation_mse": history["val_mse"][-1],
        "final_train_mr_stft": history["train_perceptual"][-1],
        "final_validation_mr_stft": history["val_perceptual"][-1],
        "compression_rate": COMPRESSION_RATE,
        "segment_duration_seconds": 1,
        "sample_rate": 16000,
        "epochs": num_epochs,
        "checkpoint": config["checkpoint"]
    })

summary_df = pd.DataFrame(summary)
summary_df.to_csv(os.path.join(RESULTS_DIR, "training_summary.csv"), index=False)
summary_df

# Listen to the original audio and reconstructions

The two selected checkpoints (baseline and improved) are evaluated on the same five consecutive one-second segments from one validation recording. The original and each reconstruction use exactly the same five-second excerpt.

In [ ]:
from IPython.display import Audio, display

# Select one validation recording and apply the same audio preprocessing.
reference_row = val_annotations.iloc[0]
reference_path = os.path.join(AUDIO_DIR, str(reference_row["mp3_path"]))
audio, sr = torchaudio.load(reference_path)

if audio.shape[0] > 1:
    audio = audio.mean(dim=0, keepdim=True)
if sr != 16000:
    audio = torchaudio.transforms.Resample(sr, 16000)(audio)

audio = audio.squeeze(0)
peak = audio.abs().max()
if peak > 0:
    audio = audio / peak

# Select five consecutive seconds within the first 27 seconds.
segment_samples = 16000
max_start = min(max(0, len(audio) // segment_samples - 5), 22)
start_second = int(torch.randint(0, max_start + 1, (1,)).item())
start = start_second * segment_samples
original = audio[start:start + 5 * segment_samples]

if len(original) < 5 * segment_samples:
    original = F.pad(original, (0, 5 * segment_samples - len(original)))

original_segments = original.split(segment_samples)
print("Reference track:", reference_row["clip_id"])
print("MTAT tag:", reference_row["natural_label"])
print("Start time:", start_second, "seconds")
print("\nORIGINAL AUDIO — 5 SECONDS")
display(Audio(original.cpu().numpy(), rate=16000))

# Reconstruct the identical five segments with both selected checkpoints.
for model_name, result in all_results.items():
    model = result["model"]
    model.eval()
    reconstructed_segments = []

    with torch.no_grad():
        for segment in original_segments:
            model_input = segment.unsqueeze(0).unsqueeze(0).to(device)
            reconstruction, _ = model(model_input)
            reconstructed_segments.append(reconstruction.squeeze().cpu())

    reconstructed_audio = torch.cat(reconstructed_segments)
    print(f"\nRECONSTRUCTION — {model_name} — 5 SECONDS")
    display(Audio(reconstructed_audio.numpy(), rate=16000))

    torchaudio.save(
        os.path.join(RESULTS_DIR, f"audio_reference_5segments_{model_name}.wav"),
        reconstructed_audio.unsqueeze(0),
        sample_rate=16000
    )

torchaudio.save(
    os.path.join(RESULTS_DIR, "audio_reference_5segments_original.wav"),
    original.cpu().unsqueeze(0),
    sample_rate=16000
)

# Report

**Data Loading and Preprocessing**

The MagnaTagATune (MTAT) dataset is processed using on-demand loading. The recordings are split by track into training and validation subsets using an 80/20 split and a fixed random seed. A reproducible 30% subset of the training recordings is used, while the full validation subset is retained. Each recording is converted to mono, resampled to 16 kHz when necessary, normalized, and cropped to one randomly selected second from the first 27 seconds. The selected segment may vary between epochs. The DataLoaders use a batch size of 128 and two worker processes.

**Two Saved Model Checkpoints**

The baseline is a three-block convolutional autoencoder without BatchNorm, optimized with waveform MSE. Its validation loss is therefore MSE. The improved model has five encoder/decoder blocks and BatchNorm in its intermediate decoder blocks. It is optimized for 20 epochs with the weighted objective $0.1 L_{MSE}+1.0 L_{MR-STFT}$. For each run, only one checkpoint is saved: the epoch with the lowest validation loss.

All models use one-second, 16 kHz segments, a 512-dimensional bottleneck, and a fixed compression rate of 31.25:1. The same training subset, validation set, augmentation pipeline, batch size, Adam optimizer, and learning rate are used. The perceptual criterion uses mel scaling, perceptual weighting, and FFT sizes of 512, 1024, and 2048 samples.

**Evaluation**

The notebook records train and validation MSE and MR-STFT for every epoch, saves the full training histories, and reports the selected epoch and validation metrics for each checkpoint. Latent-space UMAP visualizations and listening comparisons use the two selected models. The same five consecutive one-second segments from one validation recording are used for the original audio and both reconstructions.